In [1]:
from cirq_sic import *

* Test the four tasks.
* Implement entangle and don't measure.
* Write data processor: metrics and images. Gate counts.
* 1-parameter family of experiments -> metrices and images.
* Runner.
* Get arbitrary d going. [Compare aux performance].
* Negativity measure [Identify barycenters...]
* Document.
* [Time evolution.]

In [24]:
specs = {"dataset_id": "test",
         "processor_id": "willow_pink",
         "run_type": "clean",
         "qubits": get_wh_qubits(2, "ak"),
         "n_shots": 50000,
         "optimizer": "cirq",
         "d": 2,
         "fiducial": rand_ket(2),
         "fiducial_description": "rand_ket",
         "wh_implementation": "ak"}

In [25]:
tasks = {}
for task_type in sk_ground_tasks:
    task = task_from_specs(task_type, specs)
    run_sky_ground_task(task)
    tasks[task_type] = task

2025-10-22 02:29:03 [INFO] test/d2/CharacterizeWHReferenceDeviceTask/ak/rand_ket/cirq_clean_n50k_willow_pink_q4_2-5_2-6_2: Starting task...
2025-10-22 02:29:03 [INFO] test/d2/CharacterizeWHReferenceDeviceTask/ak/rand_ket/cirq_clean_n50k_willow_pink_q4_2-5_2-6_2: Creating circuits...
2025-10-22 02:29:03 [INFO] test/d2/CharacterizeWHReferenceDeviceTask/ak/rand_ket/cirq_clean_n50k_willow_pink_q4_2-5_2-6_2: Optimizing circuits...
2025-10-22 02:29:03 [INFO] test/d2/CharacterizeWHReferenceDeviceTask/ak/rand_ket/cirq_clean_n50k_willow_pink_q4_2-5_2-6_2: Sampling...
2025-10-22 02:29:04 [INFO] test/d2/CharacterizeWHReferenceDeviceTask/ak/rand_ket/cirq_clean_n50k_willow_pink_q4_2-5_2-6_2: Processing results...
2025-10-22 02:29:04 [INFO] test/d2/CharacterizeWHReferenceDeviceTask/ak/rand_ket/cirq_clean_n50k_willow_pink_q4_2-5_2-6_2: Saving...
2025-10-22 02:29:04 [INFO] test/d2/WHPOVMOnBasisStatesTask/ak/rand_ket/cirq_clean_n50k_willow_pink_q4_2-5_2-6_2: Starting task...
2025-10-22 02:29:04 [INFO] 

In [26]:
sg_results = {k: v for d in [load_results(task)["processed_data"] for task in tasks.values()] for k, v in d.items()}

In [27]:
task = tasks[CharacterizeWHReferenceDeviceTask]
E = wh_povm(task.fiducial)
P = np.array([[(a@b).trace()/b.trace() for b in E] for a in E]).real
assert np.allclose(P, exactify(task)["P"])
assert np.linalg.norm(sg_results["P"] - P) < 1e-2

In [28]:
task = tasks[WHPOVMOnBasisStatesTask]
E = wh_povm(task.fiducial)
Pi = [np.diag(np.eye(task.d)[i]) for i in range(task.d)]
p = np.array([[(a@b).trace() for b in Pi] for a in E]).real
assert np.allclose(p, exactify(task)["p"])
assert np.linalg.norm(sg_results["p"] - p) < 1e-2

In [33]:
task = tasks[BasisMeasurementOnWHStatesTask]
E = wh_povm(task.fiducial)
Pi = [np.diag(np.eye(task.d)[i]) for i in range(task.d)]
C = np.array([[(a@b).trace()/b.trace() for b in E] for a in Pi]).real
assert np.allclose(C, exactify(task)["C"])
assert np.linalg.norm(sg_results["C"] - C) < 1e-2

AssertionError: 

In [36]:
C, exactify(task)["C"], np.array(sg_results["C"])

(array([[0.862, 0.862, 0.138, 0.138],
        [0.138, 0.138, 0.862, 0.862]]),
 array([[0.033, 0.033, 0.967, 0.967],
        [0.967, 0.967, 0.033, 0.033]], dtype=float32),
 array([[0.033, 0.032, 0.967, 0.968],
        [0.967, 0.968, 0.033, 0.032]]))

In [30]:
task = tasks[BasisMeasurementOnBasisStatesTask]
q = np.eye(task.d)
assert np.allclose(q, exactify(task)["q"])
assert np.linalg.norm(sg_results["q"] - q) < 1e-2

In [31]:
C

array([[0.862, 0.862, 0.138, 0.138],
       [0.138, 0.138, 0.862, 0.862]])

In [32]:
exactify(task)["C"]

KeyError: 'C'